# PathSnap — Step 5: Folium trip inspection

Renders the **shortest**, **median** and **longest** training trips by GPS
trajectory length, so the GPS track, the road network and the ground-truth
route can be compared by eye before any modelling.

Each map carries three toggleable layers:

| layer | style | built from |
|---|---|---|
| GPS observations | small blue `CircleMarker`s | `.track` |
| road network | thin grey `PolyLine`s | `.arcs` + `.nodes`, clipped |
| ground-truth route | thick green `PolyLine` | `.route` walked through `.nodes` |

**Clipping** uses the trip's own **projected** bounding box plus a 200 m buffer
(per-trip UTM, `geo_utils.project_local`), because the graphs reach far beyond
the driven area — up to 307k arcs — and drawing all of them would produce an
unusable page. Coordinates are converted back to lon/lat for Folium, which
only speaks WGS-84.

Maps are written to `outputs/maps/trip_XXXXXXXX.html`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import folium

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import io_utils as io
import geo_utils as geo

DATA = ROOT / "data"
OUT = ROOT / "outputs"
MAPS = OUT / "maps"
MAPS.mkdir(parents=True, exist_ok=True)

#: Clip buffer around the trip bounding box, in projected metres.
CLIP_BUFFER_M = 200.0
#: Cap on drawn GPS markers; above this the track is evenly subsampled so the
#: HTML stays openable. The cap is reported on the layer name.
MAX_MARKERS = 4000

print("maps ->", MAPS)

maps -> C:\Users\LENOVO\ML Hackathon\outputs\maps


## 1. Pick the three trips

Selection uses `outputs/eda_summary.csv` when it exists (Step 4 already measured
every trip); otherwise the lengths are recomputed here. **Training trips only.**

In [2]:
summary_path = OUT / "eda_summary.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path, dtype={"trip_id": str})
    train = summary[summary.partition == "train"][["trip_id", "gps_length_m"]]
else:
    rows = []
    for tid in io.list_trips(DATA / "train"):
        tr = io.load_track(DATA / "train" / tid / f"{tid}.track")
        lon, lat = tr.lon.to_numpy(), tr.lat.to_numpy()
        rows.append({"trip_id": tid,
                     "gps_length_m": float(np.nansum(
                         geo.haversine_m(lon[:-1], lat[:-1], lon[1:], lat[1:])))})
    train = pd.DataFrame(rows)

train = train.sort_values("gps_length_m").reset_index(drop=True)
selection = {
    "shortest": train.iloc[0],
    "median": train.iloc[len(train) // 2],
    "longest": train.iloc[-1],
}
for label, row in selection.items():
    print(f"{label:9s} trip {row.trip_id}  {row.gps_length_m / 1000:8.2f} km")

shortest  trip 00000018      6.33 km


median    trip 00000025     24.25 km
longest   trip 00000054    169.25 km


## 2. Map builder

Two details worth stating explicitly:

* The route polyline is drawn **per contiguous chain** (`io_utils.route_segments`).
  11 trips contain genuine breaks in the ground-truth chain, and drawing the
  route as one polyline would invent a straight line across each gap.
* An arc is kept if **either** endpoint falls inside the buffered box, so arcs
  leaving the area are not cut off mid-way.

In [3]:
def clipped_arcs(trip: dict, proj: dict, buffer_m: float = CLIP_BUFFER_M) -> np.ndarray:
    """Arc ids whose endpoints touch the trip bounding box + buffer.

    The box is computed in projected metres from the GPS track, then arcs are
    tested in the same CRS; drawing happens later in lon/lat.
    """
    x, y = proj["track_x"], proj["track_y"]
    x0, x1 = x.min() - buffer_m, x.max() + buffer_m
    y0, y1 = y.min() - buffer_m, y.max() + buffer_m

    a = trip["arcs"]["from_node"].to_numpy()
    b = trip["arcs"]["to_node"].to_numpy()
    nx_, ny_ = proj["node_x"], proj["node_y"]
    inside = lambda px, py: (px >= x0) & (px <= x1) & (py >= y0) & (py <= y1)
    keep = inside(nx_[a], ny_[a]) | inside(nx_[b], ny_[b])
    return np.flatnonzero(keep)


def build_map(trip: dict, label: str) -> folium.Map:
    """Render one trip: GPS points, clipped road network, ground-truth route."""
    track, nodes, arcs, route = (
        trip["track"], trip["nodes"], trip["arcs"], trip["route"]
    )
    trip_id = trip["trip_id"]
    proj = geo.project_trip(trip)

    lat = track["lat"].to_numpy()
    lon = track["lon"].to_numpy()
    fmap = folium.Map(
        location=[float(lat.mean()), float(lon.mean())],
        zoom_start=13,
        tiles="OpenStreetMap",
        control_scale=True,
    )
    fmap.fit_bounds([[lat.min(), lon.min()], [lat.max(), lon.max()]])

    node_lat = nodes["lat"].to_numpy()
    node_lon = nodes["lon"].to_numpy()

    # --- road network (grey), drawn as ONE multi-polyline for speed --------
    keep = clipped_arcs(trip, proj)
    a = arcs["from_node"].to_numpy()[keep]
    b = arcs["to_node"].to_numpy()[keep]
    net_coords = [
        [[float(node_lat[i]), float(node_lon[i])], [float(node_lat[j]), float(node_lon[j])]]
        for i, j in zip(a, b)
    ]
    net_layer = folium.FeatureGroup(
        name=f"road network ({len(keep):,} of {len(arcs):,} arcs, bbox + {CLIP_BUFFER_M:g} m)"
    )
    folium.PolyLine(net_coords, color="#888888", weight=1, opacity=0.6).add_to(net_layer)
    net_layer.add_to(fmap)

    # --- ground-truth route (green), one polyline per contiguous chain -----
    chains = io.route_segments(route, arcs)
    route_layer = folium.FeatureGroup(
        name=f"ground-truth route ({route.size} arcs, {len(chains)} chain(s))"
    )
    for chain in chains:
        chain = np.asarray(chain)
        walk = [arcs["from_node"].to_numpy()[chain[0]]]
        walk.extend(arcs["to_node"].to_numpy()[chain].tolist())
        coords = [[float(node_lat[n]), float(node_lon[n])] for n in walk]
        folium.PolyLine(coords, color="#1a9850", weight=6, opacity=0.85).add_to(route_layer)
    route_layer.add_to(fmap)

    # --- GPS observations (blue) ------------------------------------------
    n = len(track)
    if n > MAX_MARKERS:
        sel = np.linspace(0, n - 1, MAX_MARKERS).astype(int)
        gps_name = f"GPS observations ({MAX_MARKERS:,} of {n:,}, subsampled)"
    else:
        sel = np.arange(n)
        gps_name = f"GPS observations ({n:,})"
    gps_layer = folium.FeatureGroup(name=gps_name)
    t = track["t_sec"].to_numpy()
    for i in sel:
        folium.CircleMarker(
            location=[float(lat[i]), float(lon[i])],
            radius=2,
            color="#2166ac",
            fill=True,
            fill_opacity=0.8,
            weight=0,
            popup=f"point_idx {int(i)} · t={t[i]:.0f}s",
        ).add_to(gps_layer)
    gps_layer.add_to(fmap)

    folium.LayerControl(collapsed=False).add_to(fmap)
    title = (
        f'<div style="position:fixed;top:10px;left:60px;z-index:9999;'
        f'background:white;padding:6px 10px;border:1px solid #999;'
        f'font:13px sans-serif">trip <b>{trip_id}</b> ({label}) &middot; '
        f'{n:,} GPS pts &middot; {route.size} route arcs &middot; EPSG:{proj["epsg"]}</div>'
    )
    fmap.get_root().html.add_child(folium.Element(title))
    return fmap

## 3. Render and save

In [4]:
saved = []
for label, row in selection.items():
    trip = io.load_trip(DATA / "train" / row.trip_id)
    fmap = build_map(trip, label)
    path = MAPS / f"trip_{row.trip_id}.html"
    fmap.save(str(path))
    saved.append((label, row.trip_id, path, path.stat().st_size / 1e6))
    print(f"{label:9s} trip {row.trip_id} -> {path.name}  ({path.stat().st_size / 1e6:.1f} MB)")

shortest  trip 00000018 -> trip_00000018.html  (0.6 MB)


median    trip 00000025 -> trip_00000025.html  (4.9 MB)


longest   trip 00000054 -> trip_00000054.html  (8.9 MB)


## 4. Open the maps

The saved HTML files are the deliverable. They are not inlined here: the
median map is ~5 MB and the longest ~9 MB, which would make this notebook
unopenable. Open them from `outputs/maps/`.

In [5]:
for label, trip_id, path, size_mb in saved:
    print(f"{label:9s} {path}  ({size_mb:.1f} MB)")

shortest  C:\Users\LENOVO\ML Hackathon\outputs\maps\trip_00000018.html  (0.6 MB)
median    C:\Users\LENOVO\ML Hackathon\outputs\maps\trip_00000025.html  (4.9 MB)
longest   C:\Users\LENOVO\ML Hackathon\outputs\maps\trip_00000054.html  (8.9 MB)


## 5. What to look for

* Blue points should sit **on** the green route. Where they run parallel to it
  at a constant offset, that trip has a GPS-to-network registration problem —
  Step 4 flagged `00000054` (12.7% of points within 50 m of the route) and
  `00000033` (41.0%) as exactly this case.
* Long dwells show up as a dense blue blob on a single arc — the first failure
  mode the Step 8 alignment has to survive.
* Green chains that stop and restart elsewhere are the route-chain breaks; the
  layer name reports how many chains the route splits into.